In [1]:
!pip install -q -U transformers datasets bitsandbytes trl peft huggingface_hub

In [2]:
from huggingface_hub import notebook_login
notebook_login()  

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
import logging

# 8-bit layers log a cast notice on every forward pass when training with bf16; it is harmless
logging.getLogger("bitsandbytes.autograd._functions").setLevel(logging.ERROR)

In [4]:
model_name = "meta-llama/Llama-3.2-1B-Instruct"

config_8bit = BitsAndBytesConfig(load_in_8bit=True)

model_8bit = AutoModelForCausalLM.from_pretrained(model_name,
                                                  quantization_config=config_8bit, 
                                                  device_map="auto", 
                                                  dtype=torch.float16,
                                                  trust_remote_code=True)

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.47GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [5]:
model_8bit

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear8bitLt(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear8bitLt(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear8bitLt(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear8bitLt(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear8bitLt(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear8bitLt(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear8bitLt(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm)

In [6]:
tokenizer = AutoTokenizer.from_pretrained( model_name, padding_side="left", trust_remote_code=True)


tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

### Downloading the Dataset and preparing

In [7]:
from datasets import load_dataset

In [8]:
dataset = load_dataset("MattCoddity/dockerNLcommands")
dataset

README.md:   0%|          | 0.00/1.46k [00:00<?, ?B/s]

06102023.json:   0%|          | 0.00/543k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2415 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input', 'output', 'instruction'],
        num_rows: 2415
    })
})

In [9]:
dataset["train"][0]

{'input': 'Give me a list of containers that have the Ubuntu image as their ancestor.',
 'output': "docker ps --filter 'ancestor=ubuntu'",
 'instruction': 'translate this sentence in docker command'}

In [10]:
from datasets import DatasetDict

train_test_split = dataset["train"].train_test_split(test_size=0.2, seed=42)

dataset = DatasetDict(
    {
        "train": train_test_split["train"],
        "validation": train_test_split["test"],
    })

In [11]:
dataset

DatasetDict({
    train: Dataset({
        features: ['input', 'output', 'instruction'],
        num_rows: 1932
    })
    validation: Dataset({
        features: ['input', 'output', 'instruction'],
        num_rows: 483
    })
})

#### Formatting the dataset - spliting

In [12]:
def to_chat_template(example):
  messages = [
      {"role": "system", "content": example["instruction"]},
      {"role": "user", "content": example["input"]},
      {"role": "assistant" , "content": example["output"]}
  ]
  return {"text": messages}

dataset = dataset.map(to_chat_template)
dataset
  

Map:   0%|          | 0/1932 [00:00<?, ? examples/s]

Map:   0%|          | 0/483 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input', 'output', 'instruction', 'text'],
        num_rows: 1932
    })
    validation: Dataset({
        features: ['input', 'output', 'instruction', 'text'],
        num_rows: 483
    })
})

In [13]:
dataset["train"]["text"][0]

[{'role': 'system', 'content': 'translate this sentence in docker command'},
 {'role': 'user',
  'content': 'Find the repository, tag, and ID of the images that were created before the latest nginx image.'},
 {'role': 'assistant',
  'content': 'docker images -f "before=nginx:latest" --format "{{.Repository}},{{.Tag}},{{.ID}}"'}]

#### Formatting the dataset -> inserting the tags

In [14]:
toeknizer_mistral = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.1")
toeknizer_mistral.apply_chat_template(dataset["train"]["text"][0], tokenize=False)

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  493kB            

tokenizer.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

'<s> [INST] translate this sentence in docker command\n\nFind the repository, tag, and ID of the images that were created before the latest nginx image. [/INST] docker images -f "before=nginx:latest" --format "{{.Repository}},{{.Tag}},{{.ID}}"</s>'

In [15]:
tokenizer.apply_chat_template(dataset["train"]["text"][0], tokenize=False)

'<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 09 Oct 2026\n\ntranslate this sentence in docker command<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nFind the repository, tag, and ID of the images that were created before the latest nginx image.<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\ndocker images -f "before=nginx:latest" --format "{{.Repository}},{{.Tag}},{{.ID}}"<|eot_id|>'

In [16]:
pwd

'/workspace/finetuning'

In [17]:
def apply_chat_temp(example):
    new_text = tokenizer.apply_chat_template(example["text"], tokenize=False)
    return {"text" : new_text}


dataset = dataset.map(apply_chat_temp)
dataset
    

Map:   0%|          | 0/1932 [00:00<?, ? examples/s]

Map:   0%|          | 0/483 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input', 'output', 'instruction', 'text'],
        num_rows: 1932
    })
    validation: Dataset({
        features: ['input', 'output', 'instruction', 'text'],
        num_rows: 483
    })
})

In [18]:
dataset["train"]["text"][0]

'<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 09 Oct 2026\n\ntranslate this sentence in docker command<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nFind the repository, tag, and ID of the images that were created before the latest nginx image.<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\ndocker images -f "before=nginx:latest" --format "{{.Repository}},{{.Tag}},{{.ID}}"<|eot_id|>'

#### Formatting the dataset -> tokenizing it to numbers

In [19]:
tokenizer(dataset["train"]["text"][0])

{'input_ids': [128000, 128000, 128006, 9125, 128007, 271, 38766, 1303, 33025, 2696, 25, 6790, 220, 2366, 18, 198, 15724, 2696, 25, 220, 2545, 5020, 220, 2366, 21, 271, 14372, 420, 11914, 304, 27686, 3290, 128009, 128006, 882, 128007, 271, 10086, 279, 12827, 11, 4877, 11, 323, 3110, 315, 279, 5448, 430, 1051, 3549, 1603, 279, 5652, 71582, 2217, 13, 128009, 128006, 78191, 128007, 271, 29748, 5448, 482, 69, 330, 15145, 28, 74661, 25, 19911, 1, 1198, 2293, 48319, 13, 4727, 39254, 3052, 13, 5786, 39254, 3052, 13, 926, 24275, 128009], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [20]:
def tokenizie_fn(example):
    return tokenizer(example["text"])

tokenized_dataset = dataset.map(tokenizie_fn,batched=True, remove_columns=['input', 'output', 'instruction', 'text'])
tokenized_dataset


Map:   0%|          | 0/1932 [00:00<?, ? examples/s]

Map:   0%|          | 0/483 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 1932
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 483
    })
})

## data collator for leveling and padding

In [21]:
from transformers import DataCollatorForLanguageModeling

In [22]:
data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False, return_tensors="pt")

In [23]:
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token

'<|eot_id|>'

### LORA 

In [24]:
from peft import LoraConfig, get_peft_model
import copy

model_8bit_clone = copy.deepcopy(model_8bit)

lora_config = LoraConfig(
    r=32, # rank of the matrix, more the r value , more params to train
    lora_alpha= 32, # the amount of influence lora matrix gonna have, higher the value, higher the influence
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"], 
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM" 
)

model_8bit_lora = get_peft_model(model_8bit_clone, lora_config)
model_8bit_lora.print_trainable_parameters()

trainable params: 22,544,384 || all params: 1,258,358,784 || trainable%: 1.7916


In [25]:
model_8bit

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear8bitLt(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear8bitLt(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear8bitLt(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear8bitLt(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear8bitLt(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear8bitLt(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear8bitLt(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm)

In [26]:
model_8bit_lora

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 2048)
        (layers): ModuleList(
          (0-15): 16 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear8bitLt(
                (base_layer): Linear8bitLt(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj):

### Setting up the hyper params for the training

In [30]:
from trl import SFTTrainer, SFTConfig

training_arguments = SFTConfig(
    output_dir="./results",
    dataset_kwargs={"skip_prepare_dataset": True}, # dataset is already tokenized, stops SFTTrainer adding an unpadded "labels" column
    per_device_train_batch_size=5,
    per_device_eval_batch_size=5,
    eval_steps=10,
    eval_strategy="steps", # Updated from deprecated evaluation_strategy
    save_steps=100,
    save_strategy="steps",
    num_train_epochs=5,
    #max_steps=60,
    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_steps=5,
    bf16=True, # model weights load as bfloat16, fp16 grad scaling fails on them
    gradient_accumulation_steps=4,
    optim="adamw_torch",
    logging_steps=10,
    report_to="none"
)
training_arguments

SFTConfig(output_dir='./results', per_device_train_batch_size=5, num_train_epochs=5, max_steps=-1, learning_rate=3e-05, lr_scheduler_type=<SchedulerType.LINEAR: 'linear'>, lr_scheduler_kwargs=None, warmup_steps=5, optim=<OptimizerNames.ADAMW_TORCH: 'adamw_torch'>, optim_args=None, weight_decay=0.01, adam_beta1=0.9, adam_beta2=0.999, adam_epsilon=1e-08, optim_target_modules=None, gradient_accumulation_steps=4, average_tokens_across_devices=True, max_grad_norm=1.0, label_smoothing_factor=0.0, bf16=True, fp16=False, bf16_full_eval=False, fp16_full_eval=False, tf32=None, gradient_checkpointing=True, gradient_checkpointing_kwargs=None, torch_compile=False, torch_compile_backend=None, torch_compile_mode=None, use_liger_kernel=False, liger_kernel_config=None, use_cache=False, neftune_noise_alpha=None, torch_empty_cache_steps=None, auto_find_batch_size=False, logging_strategy=<IntervalStrategy.STEPS: 'steps'>, logging_steps=10, logging_first_step=False, log_on_each_node=True, logging_nan_inf_f

In [31]:
trainer = SFTTrainer(
    model=model_8bit_lora,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    args=training_arguments,
    data_collator=data_collator,
    processing_class=tokenizer 
)

trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 128009}.


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
10,4.666647,3.590624,2.349136,0.504258,14295.000000
20,2.980531,2.266805,2.178054,0.609572,28531.000000
30,1.916137,1.533913,1.723079,0.748440,42868.000000
40,1.311436,1.154104,1.133358,0.805669,57251.000000
50,1.132912,1.047542,1.049194,0.819477,71937.000000
60,1.025060,0.981901,0.986333,0.826938,86304.000000
70,0.952385,0.936971,0.923327,0.832944,100505.000000
80,0.918141,0.898455,0.893543,0.840649,114898.000000
90,0.895859,0.862477,0.868373,0.843566,129436.000000
100,0.884191,0.827209,0.886036,0.848526,143184.000000


TrainOutput(global_step=485, training_loss=0.6920530191401846, metrics={'train_runtime': 1063.2553, 'train_samples_per_second': 9.085, 'train_steps_per_second': 0.456, 'total_flos': 4906115565465600.0, 'train_loss': 0.6920530191401846, 'epoch': 5.0})

### Merging the lora components

In [32]:
import torch

torch.set_printoptions(precision=8, sci_mode=False)

layer0 = model_8bit_lora.model.model.layers[0]

model_8bit_weights = layer0.self_attn.q_proj.weight
print("model_8bit_weights", model_8bit_weights)

model_8bit_weights Parameter containing:
Parameter(Int8Params([[-44,  16,  61,  ..., -22, -29,  50],
            [ 14,  70,  65,  ..., -39, -18,  13],
            [ 16,  14,  30,  ..., -34, -34, -24],
            ...,
            [ 17,  20,  40,  ..., -40, -15, -16],
            [ 32, -35,  50,  ..., -17, -41, -21],
            [-14, -30,  -7,  ...,  30,   5,  -2]], device='cuda:0',
           dtype=torch.int8))


In [33]:
import torch

torch.set_printoptions(precision=8, sci_mode=False)

layer0 = model_8bit_lora.model.model.layers[0]

weights_A = layer0.self_attn.q_proj.lora_A["default"].weight
print("LoRA A weights", weights_A)

LoRA A weights Parameter containing:
tensor([[-0.00135040,  0.00167847, -0.00013828,  ...,  0.00866699,
          0.01647949,  0.01354980],
        [ 0.02172852, -0.00451660, -0.01757812,  ..., -0.01477051,
         -0.01519775, -0.00592041],
        [ 0.00454712, -0.01770020,  0.00531006,  ...,  0.00411987,
         -0.02124023,  0.00531006],
        ...,
        [-0.01623535, -0.01116943, -0.01269531,  ...,  0.00610352,
          0.01043701, -0.00433350],
        [ 0.01843262,  0.01708984,  0.01782227,  ...,  0.00524902,
         -0.00265503, -0.01647949],
        [ 0.00598145, -0.00130463, -0.00933838,  ...,  0.00582886,
         -0.01177979,  0.00283813]], device='cuda:0', dtype=torch.bfloat16,
       requires_grad=True)


#### observations 
* other layers are in 8 bit , 
* lora in 32 bit
* we might have to download the orginal model and merge 

In [34]:
base_model = AutoModelForCausalLM.from_pretrained(model_name,
                                                  device_map="auto", 
                                                  trust_remote_code=True)

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

In [35]:
base_model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (ro

In [36]:
lora_path = "./lora_adapters"

model_8bit_lora.save_pretrained(lora_path)

In [37]:
from peft import PeftModel

base_model = PeftModel.from_pretrained(base_model, lora_path)
base_model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 2048)
        (layers): ModuleList(
          (0-15): 16 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear

In [38]:
base_model = base_model.merge_and_unload()
base_model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (ro

In [ ]:
base_model.push_to_hub("shankuriakose/Llama3.2-docker-training-complete")
tokenizer.push_to_hub("shankuriakose/Llama3.2-docker-training-complete")

### usage examples 

In [42]:
import torch
from transformers import pipeline

model_id = "shankuriakose/Llama3.2-docker-training"

pipe = pipeline(
    "text-generation",
    model= model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto"
               )

messages = [
    {"role": "system", "content": "You are a pirate chatbot who always responds in pirate speak!"},
    {"role": "user", "content": "Who are you?"},
]
outputs = pipe(
    messages,
    max_new_tokens=256,
)
print(outputs[0]["generated_text"][-1])

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'role': 'assistant', 'content': "Arrrr, I be the greatest pirate chatbot to ever sail the seven seas! Me name be Finley McPirateface, and I be here to swab yer digital decks and answer yer questions. I've got me eye on ye, matey, so keep yer wits about ye and don't be tryin' to swab me back! What be bringin' ye to these here waters?"}


In [44]:
# using the base model

import torch
from transformers import pipeline

model_id = "shankuriakose/Llama3.2-docker-training"

pipe = pipeline(
    "text-generation",
    model= base_model,
    torch_dtype=torch.bfloat16,
    device_map="auto", 
    tokenizer = tokenizer
               )

messages = [
    {"role": "system", "content": "You are a pirate chatbot who always responds in pirate speak!"},
    {"role": "user", "content": "Who are you?"},
]
outputs = pipe(
    messages,
    max_new_tokens=256,
)
print(outputs[0]["generated_text"][-1])

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'role': 'assistant', 'content': "Arrrr, me hearty! Yer lookin' fer me, the greatest pirate chatbot to ever sail the seven seas. Me name be Barnaby Blackheart, and I've been guardin' the seas and spinnin' tales fer many a landlubber. Me knowledge be vast, me wit be sharper than me cutlass, and me charm be as sweet as a chest overflowin' with gold doubloons. What be bringin' ye to these waters, matey?"}


#### Hyper Parameters Affecting the output

In [46]:

import torch
from transformers import pipeline

model_id = "shankuriakose/Llama3.2-docker-training"

pipe = pipeline(
    "text-generation",
    model= base_model,
    torch_dtype=torch.bfloat16,
    device_map="auto", 
    tokenizer = tokenizer
               )

messages = [
    {'role': 'system', 'content': 'translate this sentence in docker command'},
 {'role': 'user',
  'content': 'Find the repository, tag, and ID of the images that were created before the latest nginx image.'}
]
outputs = pipe(
    messages,
    max_new_tokens=256,
    temperature=1, ## 1 means high creativity
    top_p= 0.9, # it will filter the the top contributors that reach upto 90%
    top_k = 10, # it selects the top number, overlap => on the intersection
    do_sample=True,
    repetition_penalty=1.2
)
print(outputs[0]["generated_text"][-1]["content"])

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'top_p', 'top_k', 'temperature', 'repetition_penalty'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


docker images --filter "before=nginx:latest" --format "{{.Repository}},{{.Tag}},{{.ID}}"


In [48]:
dataset["train"][0]["output"]

'docker images -f "before=nginx:latest" --format "{{.Repository}},{{.Tag}},{{.ID}}"'